In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from src.preprocessing import process_store_data
from src.features import attach_store_data, make_features, make_targets

DATA_DIR = '../datasets/rossmann-store-sales'
STORE_FILE = os.path.join(DATA_DIR, 'store.csv')
TRAIN_FILE = os.path.join(DATA_DIR, 'train.csv')
TEST_FILE = os.path.join(DATA_DIR, 'test.csv')

FORECAST_HORIZON = 6*7 # We're forecasting daily for 6 weeks into the future
LAGS = [1, 2, 7, 30, 90, 182, 365]
DIFFS = [1, 30, 90, 182, 365]
ROLL_WINDOWS = { 7: 1, # window: lags
                30: [30, 90, 182, 365]}


# NOTE: train_df['Open'] == 0 -> train_df['Sales'] = 0. This happens always

In [25]:
df_train = pd.read_csv(TRAIN_FILE, parse_dates=['Date'], dtype={'StateHoliday': str}).drop(['Customers'], axis=1)

In [23]:
store_df = pd.read_csv(STORE_FILE)
store_df = process_store_data(store_df)

df_train = pd.read_csv(TRAIN_FILE,
                       parse_dates=['Date'],
                       dtype={'StateHoliday': str} # 0 -> '0'
                       ).drop(['Customers'], axis=1)

df_train_store = attach_store_data(df_train, store_df)

# TODO: Predict log-transformed sales?
df_features = make_features(df_train_store, lags=LAGS, roll_windows=ROLL_WINDOWS, diffs=DIFFS)
#targets = make_targets(df=df_train[['Date', 'Store', 'Sales']], horizon=FORECAST_HORIZON)

#test_df = pd.read_csv(TEST_FILE, index_col=0, parse_dates=['Date'])
#test_df = features.attach_store_data(test_df, store_df)

C:\Users\Miltos.KALIKATZAR\AppData\Local\Temp\ipykernel_18748\2097055179.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv(TRAIN_FILE, parse_dates=['Date'], dtype={'SchoolHoliday': str}).drop(['Customers'], axis=1)


In [ ]:
df = df_train_store[['Date', 'Store', 'StateHoliday']]
offsets = [-1, 0, 1] # weeks


holiday_colname = df.columns[2]
df_p = (pd.pivot(df, index=df.columns[0], columns=df.columns[1], values=holiday_colname)
        .sort_index())  # Sorted from oldest to newest

id_name = df_p.index.name
melt_index = [df_p.index.name, df_p.columns.name]
melt = lambda df, name: (df.reset_index()
                            .melt(id_vars=[id_name], value_name=name)
                            .set_index(melt_index))

# Count number of holidays per week
isodates = df_p.index.isocalendar()
week_year = isodates.year.astype(str) + isodates.week.astype(str).str.zfill(2)
num_holidays_per_week = (df_p
                         .groupby(week_year)
                         .transform(lambda group: (group != '0').sum(axis=0)))

holiday_counters = []
for week_offset in [-1, 0, 1]:
    holiday_df = num_holidays_per_week.shift(freq=pd.DateOffset(weeks=week_offset))
    holiday_df_melt = melt(holiday_df, name = f'num_{holiday_colname}_week_lag_{week_offset}')
    holiday_counters.append(holiday_df_melt)
holiday_counters = pd.concat(holiday_counters, axis=1).fillna(0).astype(int)

In [305]:
holiday_counters

,,num_StateHoliday_week_lag_-1,num_StateHoliday_week_lag_0,num_StateHoliday_week_lag_1
Date,Store,,,
2012-12-25,1,1,0,0
2012-12-26,1,1,0,0
2012-12-27,1,1,0,0
2012-12-28,1,1,0,0
2012-12-29,1,1,0,0
...,...,...,...,...
2015-08-03,1115,0,0,0
2015-08-04,1115,0,0,0
2015-08-05,1115,0,0,0
